# Improved Hybrid CNN-LSTM for Fruit Freshness Classification
## Pattern Recognition and Applications — Term Project (Improvement Notebook)

**Author:** Melih Can KÖK  
**Reference paper:** Ghosh & Singh, *Discover Artificial Intelligence* (2026), DOI: 10.1007/s44163-025-00750-7

---
### Improvements over the baseline reproduction:
| Change | Baseline (Reproduction) | This Notebook |
|--------|------------------------|---------------|
| Hybrid CNN backbone | Custom CNN from scratch (16→32→64→128) | **Pre-trained EfficientNetB0** (ImageNet) |
| LSTM type | Unidirectional LSTM (64 units) | **Bidirectional LSTM — Bi-LSTM (128 units)** |
| Sequence augmentation | Independent random per timestep → collapse | **Structured progressive augmentation** |
| LR scheduler | ReduceLROnPlateau (patience=2) | **Cosine Annealing** |
| Epochs | 30 | **50** (+ early stopping patience=10) |
| Standalone LR issue | Fixed LR 0.001 (VGG16/EffNet failed) | **Warm-up + per-model LR** |

### Literature basis:
- **Pre-trained backbone in hybrid**: Jahan et al. [22] showed pre-trained CNNs significantly improve freshness accuracy. The paper's own ablation confirmed pre-trained > custom CNN.
- **Bidirectional LSTM**: Ahmad et al. [25] demonstrated Bi-GRU outperforms unidirectional GRU for sequential classification by capturing both past and future context.
- **Structured augmentation**: The baseline collapse happened because each sequence timestep received an *independent* random transform, giving the LSTM incoherent input. Progressive augmentation creates a learnable pseudo-ripening signal.

# SECTION 1 — Libraries, GPU Setup & Data Split
(Identical to baseline reproduction notebook)

In [ ]:
import os, time, shutil, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Bidirectional,
    TimeDistributed, GlobalAveragePooling2D,
    Conv2D, MaxPooling2D, BatchNormalization, Flatten
)
from tensorflow.keras.applications import (
    InceptionV3, VGG16, ResNet50, DenseNet121, EfficientNetB0
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import kagglehub
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive

# Google Drive Baglantisi
drive.mount('/content/drive')

# Drive yolu yedeklemesi tanimlamasi
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Fruit_Freshness_Project_Results'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# GPU hafizasi
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU aktif {[g.name for g in gpus]}")

print("\n dataset yukleniyor")
dataset_path = kagglehub.dataset_download('sriramr/fruits-fresh-and-rotten-for-classification')
BASE_DIR = os.path.join(dataset_path, 'dataset')

CLASS_NAMES = ['freshapples', 'freshbanana', 'freshoranges',
               'rottenapples', 'rottenbanana', 'rottenoranges']
LABEL_MAP = {c: i for i, c in enumerate(CLASS_NAMES)}

print("Goruntu yollarının toplanmasi")
all_paths, all_labels = [], []
for split_dir in ['train', 'test']:
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(BASE_DIR, split_dir, cls)
        if not os.path.isdir(cls_dir): continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(os.path.join(cls_dir, fname))
                all_labels.append(LABEL_MAP[cls])

all_paths, all_labels = np.array(all_paths), np.array(all_labels)

# 70/10/20 oranlandi
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.667, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

WORK_DIR = './working_improved'
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(WORK_DIR, s) for s in ['train', 'val', 'test']]

def write_dirs(paths, labels, base):
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(base, cls), exist_ok=True)
    for p, lbl in zip(paths, labels):
        shutil.copy2(p, os.path.join(base, CLASS_NAMES[lbl], os.path.basename(p)))

print("dosya yapısı")
write_dirs(X_train, y_train, TRAIN_DIR)
write_dirs(X_val,   y_val,   VAL_DIR)
write_dirs(X_test,  y_test,  TEST_DIR)
print("tamamlandi")

# SECTION 2 — Hyperparameters & Generators

In [ ]:
BATCH_SIZE = 64
EPOCHS     = 50   # Paper standard (baseline used only 30)
SEQ_LEN    = 5
NUM_CLASSES = 6

# ----- Augmentation for standalone pretrained models -----
train_aug = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.10,
    brightness_range=[0.75, 1.25],
    fill_mode='nearest'
)
val_test_gen_base = ImageDataGenerator(rescale=1.0/255)

# ----- Cosine annealing LR scheduler -----
# Improvement: smoother LR decay that avoids premature convergence
def cosine_lr(epoch, lr_max=1e-3, lr_min=1e-6, T=50):
    """Cosine annealing: decays from lr_max to lr_min over T epochs.
    Based on Loshchilov & Hutter (2017), SGDR: Stochastic Gradient Descent
    with Warm Restarts."""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / T))

def get_callbacks_standalone(model_name):
    return [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=0),
        LearningRateScheduler(lambda ep: cosine_lr(ep), verbose=0)
    ]

def get_callbacks_hybrid():
    return [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=0),
        LearningRateScheduler(lambda ep: cosine_lr(ep, lr_max=5e-4), verbose=0)
    ]

def make_gen(datagen, directory, img_size=(224, 224), shuffle=True):
    return datagen.flow_from_directory(
        directory, target_size=img_size, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=shuffle, seed=42)

model_results    = {}
model_histories  = {}
model_preds      = {}
timing_results   = []

print("Config ready.")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}")

# SECTION 3 — Standalone Pre-trained Models (50 epochs, Cosine LR)

In [ ]:
def build_standalone(backbone_cls, input_shape):
    """Pre-trained CNN with custom classification head.
    Architecture matches Ghosh & Singh (2026) Table 6."""
    base = backbone_cls(
        include_top=False, weights='imagenet', input_shape=input_shape)

    # Phase 1: freeze all base layers
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(base.input, out)
    model.compile(
        optimizer=Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'])
    return model, base


BACKBONE_CFGS = {
    'InceptionV3':    {'cls': InceptionV3,    'size': (299, 299)},
    'VGG16':          {'cls': VGG16,          'size': (224, 224)},
    'ResNet50':       {'cls': ResNet50,       'size': (224, 224)},
    'DenseNet121':    {'cls': DenseNet121,    'size': (224, 224)},
    'EfficientNetB0': {'cls': EfficientNetB0, 'size': (224, 224)},
}

print("Standalone model builder ready.")

In [ ]:
for name, cfg in BACKBONE_CFGS.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print('='*60)
    tf.keras.backend.clear_session(); gc.collect()

    img_size = cfg['size']
    tr_g = make_gen(train_aug, TRAIN_DIR, img_size=img_size)
    va_g = make_gen(val_test_gen_base, VAL_DIR, img_size=img_size, shuffle=False)
    te_g = make_gen(val_test_gen_base, TEST_DIR, img_size=img_size, shuffle=False)

    model, base = build_standalone(cfg['cls'], (img_size[0], img_size[1], 3))

    # --- Phase 1: Train top layers only (10 epochs with frozen base) ---
    phase1_hist = model.fit(
        tr_g, validation_data=va_g,
        epochs=10,
        callbacks=[
            EarlyStopping('val_loss', patience=5, restore_best_weights=True)
        ],
        verbose=1
    )

    # --- Phase 2: Fine-tune last 3 conv blocks with lower LR ---
    # Improvement: consistent 2-phase training fixes VGG16 / EfficientNetB0 collapse
    base.trainable = True
    # Freeze all but last 30 layers (last ~3 blocks for most backbones)
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(1e-5),   # Low LR for fine-tuning
        loss='categorical_crossentropy',
        metrics=['accuracy'])

    t0 = time.time()
    hist = model.fit(
        tr_g, validation_data=va_g,
        epochs=EPOCHS,
        callbacks=get_callbacks_standalone(name),
        verbose=1
    )
    epoch_time = (time.time() - t0) / max(len(hist.epoch), 1)

    # Combine histories for plotting
    combined_hist = {}
    for k in hist.history:
        combined_hist[k] = phase1_hist.history.get(k, []) + hist.history[k]
    model_histories[name] = combined_hist

    # Evaluation
    te_g.reset()
    t_inf = time.time()
    y_prob = model.predict(te_g, verbose=0)
    inf_ms = (time.time() - t_inf) / te_g.samples * 1000

    timing_results.append({
        'Model': name,
        'Training time (s/epoch)': round(epoch_time, 2),
        'Inference time (ms/image)': round(inf_ms, 2)
    })

    y_pred  = np.argmax(y_prob, axis=1)
    y_true  = te_g.classes[:len(y_pred)]

    model_results[name] = {
        'Accuracy':  accuracy_score(y_true, y_pred) * 100,
        'Precision': precision_score(y_true, y_pred, average='macro') * 100,
        'Recall':    recall_score(y_true, y_pred, average='macro') * 100,
        'F1-Score':  f1_score(y_true, y_pred, average='macro') * 100,
    }
    model_preds[name] = {'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred}

    print(f"\n  {name} Results:")
    for k, v in model_results[name].items():
        print(f"    {k}: {v:.2f}%")

# SECTION 4 — Improved Hybrid CNN-LSTM

## Key architectural changes:

### 4.1 Pre-trained EfficientNetB0 CNN backbone
Instead of the custom CNN (16→32→64→128 filters from scratch), we use EfficientNetB0 pre-trained on ImageNet as the feature extractor. This provides richer, semantically meaningful feature vectors for the LSTM, matching the paper's own finding that pre-trained > custom CNN (ablation study, Table 11).

### 4.2 Bidirectional LSTM (Bi-LSTM)
The original paper uses a unidirectional LSTM(64). We use `Bidirectional(LSTM(128))` which processes each sequence both forward and backward, capturing context from both directions — directly inspired by Ahmad et al. [25] who showed Bi-GRU outperforms standard GRU for sequential classification.

### 4.3 Structured progressive augmentation for sequences
**Root cause of baseline collapse:** the sequence generator applied an independent random transform to each of the 5 timesteps. The LSTM received 5 completely unrelated views — no temporal pattern to learn.

**Fix:** Each timestep receives a *structured*, progressively stronger augmentation:
- t=1: mild (rotation ±5°, brightness ±5%)
- t=2: moderate (rotation ±10°, brightness ±10%)
- t=3: medium (rotation ±15°, brightness ±15%)
- t=4: strong (rotation ±20°, brightness ±20%)
- t=5: full (rotation ±30°, brightness ±25%, flip, noise)

This simulates a pseudo-ripening progression: an image that gradually changes, providing a coherent temporal signal for the LSTM to model.

In [ ]:
def build_improved_hybrid():
    """
    Improved Hybrid CNN-LSTM:
      - CNN backbone: pre-trained EfficientNetB0 (frozen)
      - Temporal model: Bidirectional LSTM (128 units)
      - Head: Dense(256, relu) → Dropout(0.5) → Dense(6, softmax)

    Literature basis:
      - Pre-trained backbone: Jahan et al. [22]; paper ablation Table 11
      - Bi-LSTM: Ahmad et al. [25] (Bi-GRU outperforms unidirectional)
    """
    h, w = 224, 224

    # --- CNN Sub-model (feature extractor) ---
    cnn_base = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(h, w, 3))
    cnn_base.trainable = False   # Frozen for efficient training

    cnn_input = Input(shape=(h, w, 3))
    feat = cnn_base(cnn_input, training=False)
    feat = GlobalAveragePooling2D()(feat)  # (batch, 1280)
    cnn_model = Model(cnn_input, feat, name='efficientnet_extractor')

    # --- Sequence input ---
    seq_input = Input(shape=(SEQ_LEN, h, w, 3), name='sequence_input')

    # Apply CNN to each timestep
    seq_feat = TimeDistributed(cnn_model)(seq_input)   # (batch, SEQ, 1280)
    seq_feat = Dropout(0.3)(seq_feat)

    # Bidirectional LSTM — processes sequence forward AND backward
    bi_lstm_out = Bidirectional(
        LSTM(128, activation='tanh', return_sequences=False)
    )(seq_feat)   # (batch, 256)
    bi_lstm_out = Dropout(0.5)(bi_lstm_out)

    x = Dense(256, activation='relu')(bi_lstm_out)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(seq_input, output, name='improved_hybrid')
    model.compile(
        optimizer=Adam(5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

print("Improved hybrid model builder ready.")

In [ ]:
# ---------------------------------------------------------------
# IMPROVED SEQUENCE GENERATOR — Structured Progressive Augmentation
# ---------------------------------------------------------------
# Problem with baseline: each timestep = independent random transform
#   → LSTM sees 5 unrelated images → no learnable pattern → collapse
# Fix: each timestep = deterministic, progressively stronger transform
#   → LSTM sees a mild-to-strong progression → coherent signal
# ---------------------------------------------------------------

AUGMENTERS = [
    ImageDataGenerator(  # t=1: very mild
        rotation_range=5, brightness_range=[0.95, 1.05]),
    ImageDataGenerator(  # t=2: mild
        rotation_range=10, brightness_range=[0.90, 1.10],
        horizontal_flip=True),
    ImageDataGenerator(  # t=3: moderate
        rotation_range=15, brightness_range=[0.85, 1.15],
        horizontal_flip=True, zoom_range=0.05),
    ImageDataGenerator(  # t=4: strong
        rotation_range=20, brightness_range=[0.80, 1.20],
        horizontal_flip=True, vertical_flip=True, zoom_range=0.08),
    ImageDataGenerator(  # t=5: full (same as paper)
        rotation_range=30, brightness_range=[0.75, 1.25],
        horizontal_flip=True, vertical_flip=True,
        zoom_range=0.10),
]

def structured_sequence_generator(base_gen, seq_len=SEQ_LEN, add_noise=True):
    """
    Yields (X_seq, y) where X_seq has shape (batch, seq_len, H, W, C).
    Each timestep t applies augmenter t with progressively stronger transforms,
    creating a learnable pseudo-ripening progression.
    """
    assert seq_len == len(AUGMENTERS), "seq_len must match number of augmenters"
    while True:
        for X_batch, y_batch in base_gen:
            B, H, W, C = X_batch.shape
            seq_batch = np.zeros((B, seq_len, H, W, C), dtype=np.float32)
            for t, aug in enumerate(AUGMENTERS):
                for i in range(B):
                    img_t = aug.random_transform(X_batch[i])
                    if add_noise and t == seq_len - 1:
                        # Gaussian noise only at final timestep (full augmentation)
                        noise = np.random.normal(0, 0.01, img_t.shape).astype(np.float32)
                        img_t = np.clip(img_t + noise, 0.0, 1.0)
                    seq_batch[i, t] = img_t
            yield seq_batch, y_batch


def flat_sequence_generator(base_gen, seq_len=SEQ_LEN):
    """For validation/test — all timesteps = original image (no augmentation)."""
    while True:
        for X_batch, y_batch in base_gen:
            B, H, W, C = X_batch.shape
            seq_batch = np.zeros((B, seq_len, H, W, C), dtype=np.float32)
            for t in range(seq_len):
                seq_batch[:, t, :, :, :] = X_batch
            yield seq_batch, y_batch

print("Structured sequence generators ready.")
print(f"Augmentation levels per timestep: {[f't{i+1}' for i in range(SEQ_LEN)]}")

In [ ]:
# ===== ABLATION 1: Improved Hybrid WITHOUT augmented sequences =====
# (baseline version used a custom CNN; here we use pre-trained EfficientNetB0)
# This directly tests whether the pre-trained backbone alone helps.

print("\nTraining Improved Hybrid — No Augmented Sequences (Ablation)")
tf.keras.backend.clear_session(); gc.collect()

tr_g_plain = make_gen(ImageDataGenerator(rescale=1.0/255), TRAIN_DIR, shuffle=True)
va_g_plain = make_gen(val_test_gen_base, VAL_DIR, shuffle=False)
te_g_plain = make_gen(val_test_gen_base, TEST_DIR, shuffle=False)

steps_tr   = tr_g_plain.samples  // BATCH_SIZE
steps_va   = va_g_plain.samples  // BATCH_SIZE
steps_te   = te_g_plain.samples  // BATCH_SIZE + 1

tr_seq_plain = flat_sequence_generator(tr_g_plain)
va_seq_plain = flat_sequence_generator(va_g_plain)
te_seq_plain = flat_sequence_generator(te_g_plain)

hybrid_no_aug = build_improved_hybrid()

t0 = time.time()
hist_no_aug = hybrid_no_aug.fit(
    tr_seq_plain, steps_per_epoch=steps_tr,
    validation_data=va_seq_plain, validation_steps=steps_va,
    epochs=EPOCHS, callbacks=get_callbacks_hybrid(), verbose=1)
epoch_time_h1 = (time.time() - t0) / max(len(hist_no_aug.epoch), 1)

model_histories['Improved Hybrid (No Aug)'] = hist_no_aug.history

te_g_plain.reset()
t_inf = time.time()
y_prob_h1 = hybrid_no_aug.predict(te_seq_plain, steps=steps_te, verbose=0)
inf_ms_h1 = (time.time() - t_inf) / te_g_plain.samples * 1000

timing_results.append({
    'Model': 'Improved Hybrid (No Aug)',
    'Training time (s/epoch)': round(epoch_time_h1, 2),
    'Inference time (ms/image)': round(inf_ms_h1, 2)
})

y_pred_h1 = np.argmax(y_prob_h1, axis=1)
y_true_h1 = te_g_plain.classes[:len(y_pred_h1)]

model_results['Improved Hybrid (No Aug)'] = {
    'Accuracy':  accuracy_score(y_true_h1, y_pred_h1) * 100,
    'Precision': precision_score(y_true_h1, y_pred_h1, average='macro') * 100,
    'Recall':    recall_score(y_true_h1, y_pred_h1, average='macro') * 100,
    'F1-Score':  f1_score(y_true_h1, y_pred_h1, average='macro') * 100,
}
model_preds['Improved Hybrid (No Aug)'] = {
    'y_true': y_true_h1, 'y_prob': y_prob_h1, 'y_pred': y_pred_h1}

print("\nAblation results — Improved Hybrid (No Aug):")
for k, v in model_results['Improved Hybrid (No Aug)'].items():
    print(f"  {k}: {v:.2f}%")

In [ ]:
# ===== PROPOSED IMPROVED HYBRID: EfficientNetB0 + Bi-LSTM + Structured Aug =====

print("\nTraining PROPOSED IMPROVED HYBRID")
print("Architecture: EfficientNetB0 (frozen) + Bidirectional LSTM(128) + Structured Aug")
tf.keras.backend.clear_session(); gc.collect()

# Base generators for sequence creation
tr_g_aug  = make_gen(ImageDataGenerator(rescale=1.0/255), TRAIN_DIR, shuffle=True)
va_g_eval = make_gen(val_test_gen_base, VAL_DIR, shuffle=False)
te_g_eval = make_gen(val_test_gen_base, TEST_DIR, shuffle=False)

steps_tr2 = tr_g_aug.samples  // BATCH_SIZE
steps_va2 = va_g_eval.samples // BATCH_SIZE
steps_te2 = te_g_eval.samples // BATCH_SIZE + 1

# Structured progressive sequences for training
tr_seq_aug  = structured_sequence_generator(tr_g_aug,  add_noise=True)
# Flat sequences for validation/test (no augmentation at eval time)
va_seq_flat = flat_sequence_generator(va_g_eval)
te_seq_flat = flat_sequence_generator(te_g_eval)

hybrid_proposed = build_improved_hybrid()

t0 = time.time()
hist_proposed = hybrid_proposed.fit(
    tr_seq_aug,  steps_per_epoch=steps_tr2,
    validation_data=va_seq_flat, validation_steps=steps_va2,
    epochs=EPOCHS, callbacks=get_callbacks_hybrid(), verbose=1)
epoch_time_h2 = (time.time() - t0) / max(len(hist_proposed.epoch), 1)

model_histories['Improved Hybrid (Proposed)'] = hist_proposed.history

te_g_eval.reset()
t_inf = time.time()
y_prob_h2 = hybrid_proposed.predict(te_seq_flat, steps=steps_te2, verbose=0)
inf_ms_h2 = (time.time() - t_inf) / te_g_eval.samples * 1000

timing_results.append({
    'Model': 'Improved Hybrid (Proposed)',
    'Training time (s/epoch)': round(epoch_time_h2, 2),
    'Inference time (ms/image)': round(inf_ms_h2, 2)
})

y_pred_h2 = np.argmax(y_prob_h2, axis=1)
y_true_h2 = te_g_eval.classes[:len(y_pred_h2)]

model_results['Improved Hybrid (Proposed)'] = {
    'Accuracy':  accuracy_score(y_true_h2, y_pred_h2) * 100,
    'Precision': precision_score(y_true_h2, y_pred_h2, average='macro') * 100,
    'Recall':    recall_score(y_true_h2, y_pred_h2, average='macro') * 100,
    'F1-Score':  f1_score(y_true_h2, y_pred_h2, average='macro') * 100,
}
model_preds['Improved Hybrid (Proposed)'] = {
    'y_true': y_true_h2, 'y_prob': y_prob_h2, 'y_pred': y_pred_h2}

print("\n" + "="*60)
print("PROPOSED IMPROVED HYBRID RESULTS:")
for k, v in model_results['Improved Hybrid (Proposed)'].items():
    print(f"  {k}: {v:.2f}%")

# SECTION 5 — Results, Tables & Visualizations

In [ ]:
# ===== Performance Comparison Table =====
df_all = pd.DataFrame(model_results).T.round(2)
df_timing = pd.DataFrame(timing_results).set_index('Model')

print("\n=== Model Performance Comparison ===")
display(df_all)
print("\n=== Training and Inference Times ===")
display(df_timing)

# Reference values from the original paper
paper_ref = {
    'InceptionV3':   {'Accuracy': 89.2, 'F1-Score': 88.1, 'AUC': 0.93},
    'VGG16':         {'Accuracy': 85.6, 'F1-Score': 84.2, 'AUC': 0.89},
    'ResNet50':      {'Accuracy': 91.1, 'F1-Score': 90.0, 'AUC': 0.95},
    'DenseNet121':   {'Accuracy': 91.6, 'F1-Score': 90.5, 'AUC': 0.96},
    'EfficientNetB0':{'Accuracy': 92.4, 'F1-Score': 91.4, 'AUC': 0.97},
    'Hybrid CNN-LSTM (paper)': {'Accuracy': 98.9, 'F1-Score': 97.8, 'AUC': 0.99},
}
df_paper = pd.DataFrame(paper_ref).T
print("\n=== Reference Values from Ghosh & Singh (2026) ===")
display(df_paper)

In [ ]:
# Fig. A: Accuracy comparison
models = list(df_all.index)
colors = ['#3498db']*5 + ['#f39c12', '#2ecc71']

plt.figure(figsize=(13, 6))
bars = plt.bar(models, df_all['Accuracy'], color=colors, edgecolor='black', width=0.6)
plt.axhline(y=98.9, color='red', linestyle='--', linewidth=1.5, label='Paper best (98.9%)')
plt.title('Accuracy Comparison — Improved Models vs. Paper Reference',
          fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 115)
plt.xticks(rotation=20, ha='right')
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('fig_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. B: Precision, Recall, F1 grouped bar chart
metrics = ['Precision', 'Recall', 'F1-Score']
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))
metric_colors = ['#2ecc71', '#3498db', '#e74c3c']
for i, (m, c) in enumerate(zip(metrics, metric_colors)):
    ax.bar(x + (i-1)*width, df_all[m], width, label=m, color=c,
           alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_ylim(0, 115)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Precision, Recall & F1-Score — All Improved Models',
             fontsize=14, fontweight='bold')
ax.axhline(y=97.8, color='red', linestyle='--', linewidth=1, label='Paper F1 best (97.8%)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_precision_recall_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. C: Confusion matrices for all models
n_models = len(models)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 6*nrows))
axes = axes.ravel()

for idx, m_name in enumerate(models):
    cm = confusion_matrix(
        model_preds[m_name]['y_true'],
        model_preds[m_name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(m_name, fontweight='bold', fontsize=10)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. D: ROC curves (macro-average)
y_true_bin = label_binarize(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    classes=np.arange(NUM_CLASSES))

plt.figure(figsize=(10, 8))
cmap = plt.cm.get_cmap('tab10')

for idx, m_name in enumerate(models):
    y_prob = model_preds[m_name]['y_prob']
    y_true_local = label_binarize(
        model_preds[m_name]['y_true'], classes=np.arange(NUM_CLASSES))
    fprs, tprs, roc_aucs = {}, {}, {}
    for c in range(NUM_CLASSES):
        fprs[c], tprs[c], _ = roc_curve(y_true_local[:, c], y_prob[:, c])
        roc_aucs[c] = auc(fprs[c], tprs[c])
    all_fpr = np.unique(np.concatenate([fprs[c] for c in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fprs[c], tprs[c])
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color=cmap(idx),
             label=f'{m_name} (AUC={macro_auc:.3f})', lw=1.5)

plt.plot([0,1],[0,1],'k--',lw=0.8)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — Macro Average', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. E: Training & validation loss/accuracy — Proposed Improved Hybrid
h = model_histories['Improved Hybrid (Proposed)']
epochs_x = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, h['loss'], label='Train Loss', color='blue')
axes[0].plot(epochs_x, h['val_loss'], label='Val Loss', color='red')
axes[0].set_title('Loss over Epochs — Improved Hybrid', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, h['accuracy'], label='Train Acc', color='blue')
axes[1].plot(epochs_x, h['val_accuracy'], label='Val Acc', color='red')
axes[1].set_title('Accuracy over Epochs — Improved Hybrid', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig. F: Per-class confusion matrix for Proposed Improved Hybrid
cm_hybrid = confusion_matrix(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    model_preds['Improved Hybrid (Proposed)']['y_pred'])

plt.figure(figsize=(9, 7))
sns.heatmap(cm_hybrid, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Per-class Confusion Matrix — Improved Hybrid (Proposed)',
          fontsize=13, fontweight='bold')
plt.ylabel('Actual Class'); plt.xlabel('Predicted Class')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('fig_per_class_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-class metrics
from sklearn.metrics import classification_report
print("\n=== Per-class Classification Report — Improved Hybrid ===")
print(classification_report(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    model_preds['Improved Hybrid (Proposed)']['y_pred'],
    target_names=CLASS_NAMES, digits=4))

In [ ]:
# ===== Ablation Study Summary =====
ablation = {
    'Baseline (No Aug, Custom CNN)': model_results.get('Hybrid CNN-LSTM (No Aug)',
        {'Accuracy': 97.87, 'Precision': 97.74, 'Recall': 97.80, 'F1-Score': 97.80}),
    'Improved (No Aug, EfficientNetB0)': model_results['Improved Hybrid (No Aug)'],
    'Proposed (EfficientNetB0 + Bi-LSTM + Struct.Aug)': model_results['Improved Hybrid (Proposed)'],
    'Paper Hybrid (reference)': {
        'Accuracy': 98.9, 'Precision': 97.5, 'Recall': 98.1, 'F1-Score': 97.8}
}

df_ablation = pd.DataFrame(ablation).T.round(2)
print("\n=== Ablation Study ===")
display(df_ablation)

In [ ]:
# Save the proposed hybrid model
hybrid_proposed.save('improved_hybrid_model.keras')

# List of files to back up
files_to_backup = [
    'improved_hybrid_model.keras',
    'fig_accuracy_comparison.png',
    'fig_precision_recall_f1.png',
    'fig_confusion_matrices.png',
    'fig_roc_curves.png',
    'fig_training_curves.png',
    'fig_per_class_cm.png'
]

print("Backing up results to Google Drive...")
for file in files_to_backup:
    if os.path.exists(file):
        shutil.copy(file, os.path.join(DRIVE_BACKUP_DIR, file))
        print(f"Backed up {file}")